In [2]:
# import sys
# import subprocess
# import importlib
# from IPython.display import display, Javascript

# print("[INFO] Memeriksa dan menginstal dependensi yang diperlukan pada environment kernel saat ini...")
# package_specs = [
#     'accelerate',
#     'transformers==4.46.3',
#     'datasets',
#     'evaluate',
#     'rouge-score',
#     'pyarrow',
# ]
# import_modules = [
#     'accelerate',
#     'transformers',
#     'datasets',
#     'evaluate',
#     'rouge_score',
#     'pyarrow',
# ]
# missing = []
# for module_name in import_modules:
#     try:
#         importlib.import_module(module_name)
#     except Exception:
#         missing.append(module_name)
# if missing:
#     print(f"[INFO] Paket hilang: {missing}. Menginstal...")
#     subprocess.check_call([sys.executable, '-m', 'pip', 'install', *package_specs, '-U'])
#     # Coba impor lagi setelah instalasi
#     not_loaded = []
#     for module_name in import_modules:
#         try:
#             importlib.import_module(module_name)
#         except Exception:
#             not_loaded.append(module_name)
#     if not_loaded:
#         print(f"[PERINGATAN] Beberapa paket belum dapat diimpor setelah instalasi: {not_loaded}")
#         print("Silakan restart kernel (manual) atau klik tombol di bawah untuk restart otomatis.")
#         display(Javascript('Jupyter.notebook.kernel.restart()'))
#     else:
#         print("[SUKSES] Semua paket terinstal dan dapat diimpor tanpa restart.")
# else:
#     print("[SUKSES] Semua dependensi sudah terpasang.")

In [3]:
# # Debug imports: print versions and detailed tracebacks for failures
# import traceback
# try:
#     import transformers
#     print(f"transformers version: {transformers.__version__}")
# except Exception:
#     print("Failed importing transformers:")
#     traceback.print_exc()

# # Try importing Seq2SeqTrainer and the underlying module
# try:
#     from transformers import Seq2SeqTrainer
#     print("Seq2SeqTrainer imported successfully")
# except Exception:
#     print("Failed importing Seq2SeqTrainer:")
#     traceback.print_exc()
#     try:
#         import importlib
#         importlib.import_module('transformers.trainer_seq2seq')
#         print("transformers.trainer_seq2seq module imported successfully")
#     except Exception:
#         print("Failed importing transformers.trainer_seq2seq:")
#         traceback.print_exc()

# # Print installed versions for related packages
# try:
#     import importlib.metadata as md
#     for pkg in ['transformers','accelerate','datasets','torch','evaluate','pyarrow']:
#         try:
#             print(pkg + ':', md.version(pkg))
#         except Exception as e:
#             print(pkg + ': not installed or unknown (', e, ')')
# except Exception:
#     print('Could not get package versions:', end=' ')
#     traceback.print_exc()

In [16]:
import os
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    T5Tokenizer,                  # Diubah dari AutoTokenizer ke Tokenizer khas T5
    AutoModelForSeq2SeqLM,        # Diubah dari MBart ke Model Seq2Seq umum (cocok untuk T5)
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Perangkat aktif untuk training: {device.upper()}")
if device == "cuda":
    print(f"[INFO] Menggunakan GPU: {torch.cuda.get_device_name(0)}")

[INFO] Perangkat aktif untuk training: CUDA
[INFO] Menggunakan GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [6]:
print("\n[INFO] Mengunduh data Parquet aman langsung dari repositori Hugging Face...")

# URL resmi hasil konversi otomatis Hugging Face untuk XL-Sum Indonesian
URL_TRAIN = "https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/main/data/indonesian/train-00000-of-00001.parquet"
URL_VAL = "https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/main/data/indonesian/validation-00000-of-00001.parquet"
URL_TEST = "https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/main/data/indonesian/test-00000-of-00001.parquet"


[INFO] Mengunduh data Parquet aman langsung dari repositori Hugging Face...


In [7]:
print("[INFO] Memuat dataset XL-Sum menggunakan local Parquet loader...")

# SOLUSI: Tetap menggunakan load_dataset, tetapi menggunakan driver "parquet" 
# untuk memotong skrip .py yang diblokir oleh Hugging Face.
dataset = load_dataset(
    "parquet",
    data_files={
        "train": "xlsum-train.parquet",
        "validation": "xlsum-validation.parquet",
        "test": "xlsum-test.parquet"
    }
)

[INFO] Memuat dataset XL-Sum menggunakan local Parquet loader...


In [8]:
train_ds = dataset["train"].shuffle(seed=42).select(range(3000))
val_ds = dataset["validation"].shuffle(seed=42).select(range(500))
test_ds = dataset["test"].shuffle(seed=42).select(range(500))

# NOTE AMAN: Jika file 'xlsum-test.parquet' lokal Anda sempat rusak saat diunduh, 
# Anda bisa mengambil data test murni dari index lanjutan validation dengan baris ini:
# test_ds = dataset["validation"].shuffle(seed=42).select(range(500, 1000))

print(f"[SUKSES] Dataset berhasil dimuat ke objek Hugging Face!")
print(f"-> Train Split: {len(train_ds)} baris")
print(f"-> Val Split  : {len(val_ds)} baris")
print(f"-> Test Split : {len(test_ds)} baris")

[SUKSES] Dataset berhasil dimuat ke objek Hugging Face!
-> Train Split: 3000 baris
-> Val Split  : 500 baris
-> Test Split : 500 baris


In [9]:
model_name = "google-t5/t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [10]:
def preprocess_function(examples):
    # Karakteristik Utama T5: Wajib menambahkan prefix tugas seperti "summarize: " 
    # di depan teks input agar model tahu instruksi yang harus dijalankan.
    inputs = ["summarize: " + str(text) for text in examples["text"]]
    
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("\n[INFO] Menjalankan tokenisasi dataset...")
tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_val = val_ds.map(preprocess_function, batched=True)
tokenized_test = test_ds.map(preprocess_function, batched=True)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


[INFO] Menjalankan tokenisasi dataset...


In [11]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_xlsum_indonesian",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,  # Sangat optimal untuk VRAM RTX 4060 Laptop Anda
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    logging_steps=20,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(), # Mengaktifkan presisi cepat jika GPU aktif
    logging_dir="./logs"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("[SIAP LO] Kode Anda lolos dari pemblokiran dan Trainer siap dijalankan!")
trainer.train()

C:\Users\alvin\AppData\Local\Temp\ipykernel_13576\2380924020.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


[SIAP LO] Kode Anda lolos dari pemblokiran dan Trainer siap dijalankan!


  2%|▏         | 20/1125 [00:04<04:05,  4.50it/s]

{'loss': 6.7711, 'grad_norm': 41.44258499145508, 'learning_rate': 1.969777777777778e-05, 'epoch': 0.05}


  4%|▎         | 40/1125 [00:09<04:22,  4.14it/s]

...

100%|█████████▉| 1121/1125 [03:29<00:00,  4.80it/s]

{'loss': 1.9901, 'grad_norm': 0.9893644452095032, 'learning_rate': 1.4222222222222224e-07, 'epoch': 2.99}


                                                   
100%|██████████| 1125/1125 [03:35<00:00,  5.22it/s]

{'eval_loss': 1.6842412948608398, 'eval_runtime': 3.7798, 'eval_samples_per_second': 132.281, 'eval_steps_per_second': 16.667, 'epoch': 3.0}
{'train_runtime': 215.3237, 'train_samples_per_second': 41.798, 'train_steps_per_second': 5.225, 'train_loss': 2.103468763563368, 'epoch': 3.0}


TrainOutput(global_step=1125, training_loss=2.103468763563368, metrics={'train_runtime': 215.3237, 'train_samples_per_second': 41.798, 'train_steps_per_second': 5.225, 'total_flos': 1218076213248000.0, 'train_loss': 2.103468763563368, 'epoch': 3.0})

In [12]:
print("\n--- Memulai Proses Evaluasi Menggunakan Data Test ---")

rouge = evaluate.load("rouge")

# Menggunakan trainer untuk memprediksi teks dari tokenized_test
predictions = trainer.predict(tokenized_test)

# PERBAIKAN BUG 1: Ambil indeks 0 jika predictions berupa tuple/objek kompleks
generated_tokens = predictions.predictions
if isinstance(generated_tokens, tuple):
    generated_tokens = generated_tokens[0]

# Mengambil id label asli dan membersihkan padding token (-100)
label_ids = predictions.label_ids
label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

# Decode hasil prediksi model dan label asli kembali menjadi teks kalimat biasa
# PERBAIKAN BUG 1: Menggunakan 'generated_tokens' yang sudah aman
decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

# Menghitung skor ROUGE (ROUGE-1, ROUGE-2, ROUGE-L)
result = rouge.compute(predictions=decoded_preds, references=decoded_labels)

print("\n========================================")
print("[HASIL EVALUASI AKHIR ROUGE SCORE]:")
print("========================================")
for key, value in result.items():
    print(f"{key}: {value * 100:.2f}%")
print("========================================")


--- Memulai Proses Evaluasi Menggunakan Data Test ---


c:\Users\alvin\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\generation\utils.py:1375: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
100%|██████████| 63/63 [00:16<00:00,  3.91it/s]



[HASIL EVALUASI AKHIR ROUGE SCORE]:
rouge1: 6.88%
rouge2: 1.92%
rougeL: 6.18%
rougeLsum: 6.15%


In [13]:
output_model_dir = "./t5-small-indonesian-summarizer"

print(f"\n[INFO] Menyimpan model dan tokenizer ke folder '{output_model_dir}'...")
model.save_pretrained(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

print("[SUKSES] Model Anda telah tersimpan dengan aman dan siap dipakai kapan saja!")


[INFO] Menyimpan model dan tokenizer ke folder './t5-small-indonesian-summarizer'...
[SUKSES] Model Anda telah tersimpan dengan aman dan siap dipakai kapan saja!


In [14]:
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM

# Memuat kembali model dari folder penyimpanan lokal Anda
model_path = "./t5-small-indonesian-summarizer"
my_tokenizer = T5Tokenizer.from_pretrained(model_path)
my_model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Teks berita bahasa Indonesia acak yang ingin Anda ringkas
artikel_baru = "Isi teks atau artikel berita panjang bahasa Indonesia Anda di sini..."

# Preprocessing teks khas T5
input_text = "summarize: " + artikel_baru
inputs = my_tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)

# Generate ringkasan teks menggunakan model Anda
summary_ids = my_model.generate(inputs["input_ids"], max_length=128, min_length=30, length_penalty=2.0, num_beams=4, early_stopping=True)
hasil_ringkasan = my_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("\n[Hasil Ringkasan Model Anda]:")
print(hasil_ringkasan)


[Hasil Ringkasan Model Anda]:
Isi teks atau artikel berita panjang bahasa Indonesia Anda di sini di sini...


In [17]:
print("\n[INFO] Menyusun rangkuman metrik training...")

# Mengekstrak history log dari trainer
history = trainer.state.log_history

# Satukan log training (loss) dan log evaluasi (val_loss & rouge) berdasarkan epoch
epoch_summary = {}

for log in history:
    epoch = int(log.get("epoch", 0)) if "epoch" in log else None
    if epoch is None:
        continue
        
    if epoch not in epoch_summary:
        epoch_summary[epoch] = {"Epoch": epoch}
        
    # Ambil nilai jika tersedia di dalam log
    if "loss" in log: # Training loss
        epoch_summary[epoch]["Train Loss"] = round(log["loss"], 4)
    if "eval_loss" in log: # Validation/Test Loss
        epoch_summary[epoch]["Val/Test Loss"] = round(log["eval_loss"], 4)
    if "eval_rouge1" in log:
        epoch_summary[epoch]["ROUGE-1 (%)"] = round(log["eval_rouge1"], 2)
    if "eval_rouge2" in log:
        epoch_summary[epoch]["ROUGE-2 (%)"] = round(log["eval_rouge2"], 2)
    if "eval_rougeL" in log:
        epoch_summary[epoch]["ROUGE-L (%)"] = round(log["eval_rougeL"], 2)

# Ubah dictionary menjadi Pandas DataFrame agar rapi dalam bentuk tabel
df_summary = pd.DataFrame(list(epoch_summary.values()))

# Pastikan urutan kolom rapi
kolom_ideal = ["Epoch", "Train Loss", "Val/Test Loss", "ROUGE-1 (%)", "ROUGE-2 (%)", "ROUGE-L (%)"]
kolom_ada = [col for col in kolom_ideal if col in df_summary.columns]
df_summary = df_summary[kolom_ada]

print("\n==================================================================")
print("             RANGKUMAN METRIK MODEL PER EPOCH                    ")
print("==================================================================")
print(df_summary.to_string(index=False))
print("==================================================================")

# ==============================================================================
# 5. MENYIMPAN MODEL SECARA AMAN (Ganti nama folder untuk hindari OS Error 1224)
# ==============================================================================
output_model_dir = "./t5-small-indonesian-summarizer-final"
print(f"\n[INFO] Menyimpan model akhir ke folder '{output_model_dir}'...")
model.save_pretrained(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print("[SUKSES] Semua proses selesai dan model berhasil disimpan!")


[INFO] Menyusun rangkuman metrik training...

             RANGKUMAN METRIK MODEL PER EPOCH                    
 Epoch  Train Loss  Val/Test Loss
     0      2.0164            NaN
     1      1.8550         1.7430
     2      1.9901         1.6933
     3         NaN         1.6842

[INFO] Menyimpan model akhir ke folder './t5-small-indonesian-summarizer-final'...
[SUKSES] Semua proses selesai dan model berhasil disimpan!
